<a href="https://colab.research.google.com/github/huggingface/deep-rl-class/blob/main/notebooks/unit8/unit8_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第 8 单元 第 2 部分:进阶深度强化学习。使用 Sample Factory 从像素玩转 Doom(毁灭战士)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/thumbnail2.png" alt="Thumbnail"/>

在本 notebook 中,我们将学习如何训练一个深度神经网络,在基于游戏《Doom》(毁灭战士)的 3D 环境中收集物品,下面展示了最终策略的一段视频。我们使用 [Sample Factory](https://www.samplefactory.dev/) 来训练这个策略,它是 PPO(近端策略优化)算法的一种异步实现。

请注意以下几点:

*   [Sample Factory](https://www.samplefactory.dev/) 是一个进阶的强化学习框架,**只能在 Linux 和 Mac 上运行**(不支持 Windows)。

*  该框架在**拥有众多 CPU 核心的 GPU 机器上表现最佳**,可以达到每秒 10 万次交互的速度。而标准 Colab notebook 的可用资源**会限制这个库的性能**。因此在这种环境下的速度**并不能反映其真实性能**。
* Sample Factory 在多种设置下都有基准测试结果,如果你想了解更多,请查看[示例](https://github.com/alex-petrenko/sample-factory/tree/master/sf_examples)。


In [ ]:
from IPython.display import HTML

HTML('''<video width="640" height="480" controls>
  <source src="https://huggingface.co/edbeeching/doom_health_gathering_supreme_3333/resolve/main/replay.mp4"
  type="video/mp4">Your browser does not support the video tag.</video>'''
)

要通过[认证流程](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process)的验证,你需要推送一个模型:

- `doom_health_gathering_supreme`(生命采集·至尊版场景)需取得 >= 5 的结果。

要查看你的结果,请前往[排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)找到你的模型,**结果 = 平均奖励(mean_reward)- 奖励的标准差(std of reward)**

如果你找不到自己的模型,**请转到页面底部并点击刷新按钮**

有关认证流程的更多信息,请查看这一节 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## 设置 GPU 💪
- 为了**加速智能体的训练,我们将使用 GPU**。为此,请前往 `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`(硬件加速器 > GPU)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

在开始训练智能体之前,让我们先**了解将要使用的库和环境**。

## Sample Factory

[Sample Factory](https://www.samplefactory.dev/) 是**最快的强化学习库之一,专注于以非常高效的方式同步和异步地实现策略梯度(即 PPO)**。

Sample Factory 经过**充分测试,被众多研究人员和从业者使用**,并且一直在积极维护。它的实现**能够在多种领域中达到 SOTA(最先进)性能,同时把强化学习实验的训练时间和硬件需求降到最低**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/samplefactoryenvs.png" alt="Sample factory"/>



### 主要特性

- 高度优化的算法[架构](https://www.samplefactory.dev/06-architecture/overview/),实现最大的学习吞吐量
- [同步与异步](https://www.samplefactory.dev/07-advanced-topics/sync-async/)训练模式
- [串行(单进程)模式](https://www.samplefactory.dev/07-advanced-topics/serial-mode/),便于调试
- 在基于 CPU 的环境和 [GPU 加速的环境](https://www.samplefactory.dev/09-environment-integrations/isaacgym/)中都有最佳性能
- 单智能体与多智能体训练、自博弈(self-play),支持在单个或多个 GPU 上同时[训练多个策略](https://www.samplefactory.dev/07-advanced-topics/multi-policy-training/)
- 基于种群的训练(Population-Based Training,[PBT](https://www.samplefactory.dev/07-advanced-topics/pbt/))
- 离散、连续、混合动作空间
- 基于向量、图像、字典的观测空间
- 通过解析动作/观测空间的规格自动创建模型架构。支持[自定义模型架构](https://www.samplefactory.dev/03-customization/custom-models/)
- 设计时就考虑了被导入其他项目使用,[自定义环境](https://www.samplefactory.dev/03-customization/custom-environments/)是一等公民
- 详细的 [WandB 和 Tensorboard 摘要](https://www.samplefactory.dev/05-monitoring/metrics-reference/)、[自定义指标](https://www.samplefactory.dev/05-monitoring/custom-metrics/)
- [HuggingFace 🤗 集成](https://www.samplefactory.dev/10-huggingface/huggingface/)(把训练好的模型和指标上传到 Hub)
- [多个](https://www.samplefactory.dev/09-environment-integrations/mujoco/) [示例](https://www.samplefactory.dev/09-environment-integrations/atari/) [环境](https://www.samplefactory.dev/09-environment-integrations/vizdoom/) [集成](https://www.samplefactory.dev/09-environment-integrations/dmlab/),附带调优过的参数和训练好的模型

上述所有策略都可以在 🤗 Hub 上找到。请搜索标签 [sample-factory](https://huggingface.co/models?library=sample-factory&sort=downloads)

### sample-factory 的工作原理

Sample-factory 是**社区可用的优化程度最高的强化学习实现之一**。

它的工作方式是**启动多个进程,分别运行 rollout worker(采样工作进程)、inference worker(推理工作进程)和一个 learner worker(学习工作进程)**。

这些 *worker* **通过共享内存进行通信,从而降低了进程间的通信开销**。

*rollout worker* 与环境交互,并把观测发送给 *inference worker*。

*inference worker* 查询策略的一个固定版本,并**把动作发回给 rollout worker**。

经过 *k* 步之后,rollout worker 会把一条经验轨迹发送给 learner worker,**后者利用它来更新智能体的策略网络**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/samplefactory.png" alt="Sample factory"/>

### Sample-factory 中的 Actor-Critic 模型

Sample Factory 中的 Actor-Critic 模型由三个组件构成:

- **Encoder(编码器)** - 处理输入观测(图像、向量)并把它们映射成一个向量。这是模型中最可能需要你自定义的部分。
- **Core(核心)** - 整合来自一个或多个编码器的向量;在基于记忆的智能体中,可以选择性地包含单层或多层 LSTM/GRU。
- **Decoder(解码器)** - 在计算策略输出和价值输出之前,对模型核心的输出再施加额外的层。

这个库经过设计,可以自动支持任何观测空间和动作空间。用户可以轻松添加自己的自定义模型。更多信息请参阅[文档](https://www.samplefactory.dev/03-customization/custom-models/#actor-critic-models-in-sample-factory)。

## ViZDoom

[ViZDoom](https://vizdoom.cs.put.edu.pl/) 是 **Doom 引擎的开源 Python 接口**。

该库由 Marek Wydmuch 和 Michal Kempka 于 2016 年在波兰波兹南理工大学(Poznan University of Technology)计算科学研究所创建。

这个库让我们能够**在多种场景中直接基于屏幕像素训练智能体**,包括下面视频中展示的团队死斗模式(team deathmatch)。由于 ViZDoom 环境基于一款诞生于 90 年代的游戏,它可以在现代硬件上加速运行,**让我们能够相当快速地学习复杂的 AI 行为**。

该库包含的特性包括:

- 跨平台(Linux、macOS、Windows),
- 提供 Python 和 C++ API,
- [OpenAI Gym](https://www.gymlibrary.dev/) 环境包装器
- 易于创建自定义场景(提供可视化编辑器、脚本语言和示例),
- 异步与同步的单人和多人模式,
- 轻量(仅几 MB)且快速(同步模式下单线程最高可达 7000 fps),
- 可自定义分辨率和渲染参数,
- 可访问深度缓冲区(3D 视觉),
- 自动标注画面中可见的游戏物体,
- 可访问音频缓冲区
- 可访问角色/物体列表和地图几何信息,
- 离屏渲染与回合(episode)录制,
- 异步模式下的时间缩放。

## 我们首先需要安装 ViZDoom 环境所需的一些依赖

Colab 运行时设置好之后,我们可以先安装 Linux 上运行 ViZDoom 所需的依赖。

如果你是在自己的 Mac 电脑上跟随操作,请按照[github 页面](https://github.com/Farama-Foundation/ViZDoom/blob/master/doc/Quickstart.md#-quickstart-for-macos-and-anaconda3-python-36)上的安装说明进行。

In [ ]:
%%capture
%%bash
# 从下方地址安装 ViZDoom 的依赖
# https://github.com/mwydmuch/ViZDoom/blob/master/doc/Building.md#-linux

apt-get install build-essential zlib1g-dev libsdl2-dev libjpeg-dev \
nasm tar libbz2-dev libgtk2.0-dev cmake git libfluidsynth-dev libgme-dev \
libopenal-dev timidity libwildmidi-dev unzip ffmpeg

# Boost 库
apt-get install libboost-all-dev

# Lua 绑定的依赖
apt-get install liblua5.1-dev

## 接下来我们就可以安装 Sample Factory 和 ViZDoom 了
- 这可能需要 7 分钟

In [ ]:
# 安装 Python 库
# 感谢 toinsson
!pip install faster-fifo==1.4.2
!pip install vizdoom

In [ ]:
!pip install sample-factory==2.1.1

## 在 sample-factory 中设置 Doom 环境

In [ ]:
import functools

from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.train import run_rl

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec


# 注册所有 ViZDoom 环境
def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)

# Sample Factory 允许注册自定义的神经网络架构
# 更多细节参见 https://github.com/alex-petrenko/sample-factory/blob/master/sf_examples/vizdoom/doom/doom_model.py
def register_vizdoom_models():
    global_model_factory().register_encoder_factory(make_vizdoom_encoder)


def register_vizdoom_components():
    register_vizdoom_envs()
    register_vizdoom_models()

# 解析命令行参数并创建配置
def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    # Doom 环境特有的参数
    add_doom_env_args(parser)
    # 覆盖算法参数的 Doom 默认值
    doom_override_defaults(parser)
    # 第二轮解析生成最终配置
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg

设置完成后,我们就可以训练智能体了。这里我们选择学习一个名为 `Health Gathering Supreme`(生命采集·至尊版)的 ViZDoom 任务。

### 场景(scenario):Health Gathering Supreme(生命采集·至尊版)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/Health-Gathering-Supreme.png" alt="Health-Gathering-Supreme"/>



这个场景的目标是**教会智能体如何在不知道什么能让它存活的前提下生存下来**。智能体只知道**生命是宝贵的**、死亡是不好的,因此**它必须学会什么能延长自己的存活时间,并意识到自己的生命值与此息息相关**。

地图是一个矩形,四周有围墙,地面是绿色的酸性地面,会**周期性地对玩家造成伤害**。初始时,地图上均匀分布着一些医疗包(medkit),并且每隔一段时间就会有一个新的医疗包从天而降。**医疗包可以恢复玩家的一部分生命值** —— 智能体要想生存就必须拾取它们。回合(episode)在玩家死亡或超时后结束。

更多配置:
- Living_reward(存活奖励)= 1
- 3 个可用按钮:左转、右转、前进
- 1 个可用的游戏变量:HEALTH(生命值)
- 死亡惩罚 = 100

你可以在[这里](https://github.com/Farama-Foundation/ViZDoom/tree/master/scenarios)进一步了解 ViZDoom 中可用的场景。

还有一些为 ViZDoom 创建的更复杂的场景,例如[这个 github 页面](https://github.com/edbeeching/3d_control_deep_rl)上详细介绍的那些。



## 训练智能体
- 我们将训练智能体 4000000 步,大约需要 20 分钟

In [ ]:
## 开始训练,这大约需要 15 分钟
register_vizdoom_components()

# 我们今天训练的场景是 health gathering(生命采集)
# 其他场景包括 "doom_basic"、"doom_two_colors_easy"、"doom_dm"、"doom_dwango5"、"doom_my_way_home"、"doom_deadly_corridor"、"doom_defend_the_center"、"doom_defend_the_line"
env = "doom_health_gathering_supreme"
cfg = parse_vizdoom_cfg(argv=[f"--env={env}", "--num_workers=8", "--num_envs_per_worker=4", "--train_for_env_steps=4000000"])

status = run_rl(cfg)

## 让我们来看看训练出的策略的表现,并输出一段智能体的视频。

In [ ]:
from sample_factory.enjoy import enjoy
cfg = parse_vizdoom_cfg(argv=[f"--env={env}", "--num_workers=1", "--save_video", "--no_render", "--max_num_episodes=10"], evaluation=True)
status = enjoy(cfg)

## 现在让我们把智能体的表现可视化

In [ ]:
from base64 import b64encode
from IPython.display import HTML

mp4 = open('/content/train_dir/default_experiment/replay.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

智能体已经学到了一些东西,但它的表现还可以更好。显然我们需要训练更长的时间。不过,让我们先把这个模型上传到 Hub。

## 现在让我们把你的 checkpoint(检查点)和视频上传到 Hugging Face Hub




要与社区分享你的模型,还需要完成以下三个步骤:

1️⃣ (如果还没有的话)注册一个 HF 账号 ➡ https://huggingface.co/join

2️⃣ 登录之后,你需要保存来自 Hugging Face 网站的认证令牌(token)。
- 创建一个新令牌(https://huggingface.co/settings/tokens),**角色需选择 write(写权限)**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制该令牌
- 运行下面的单元格并粘贴令牌

如果你不想使用 Google Colab 或 Jupyter Notebook,就需要改用这个命令:`huggingface-cli login`

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
!git config --global credential.helper store

In [ ]:
from sample_factory.enjoy import enjoy

hf_username = "ThomasSimonini" # 在这里填入你的 Hugging Face 用户名

cfg = parse_vizdoom_cfg(argv=[f"--env={env}", "--num_workers=1", "--save_video", "--no_render", "--max_num_episodes=10", "--max_num_frames=100000", "--push_to_hub", f"--hf_repository={hf_username}/rl_course_vizdoom_health_gathering_supreme"], evaluation=True)
status = enjoy(cfg)

## 让我们加载另一个模型




这个智能体的表现不错,但还能更好!让我们从 Hub 下载并可视化一个训练了 100 亿(10B)时间步的智能体。

In [ ]:
# 从 Hub 下载智能体
!python -m sample_factory.huggingface.load_from_hub -r edbeeching/doom_health_gathering_supreme_2222 -d ./train_dir


In [ ]:
!ls train_dir/doom_health_gathering_supreme_2222

In [ ]:
env = "doom_health_gathering_supreme"
cfg = parse_vizdoom_cfg(argv=[f"--env={env}", "--num_workers=1", "--save_video", "--no_render", "--max_num_episodes=10", "--experiment=doom_health_gathering_supreme_2222", "--train_dir=train_dir"], evaluation=True)
status = enjoy(cfg)

In [ ]:
mp4 = open('/content/train_dir/doom_health_gathering_supreme_2222/replay.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

## 更多附加挑战 🏆:Doom 死斗模式(Deathmatch)

训练一个智能体来玩 Doom 死斗模式(deathmatch),**需要一台比 Colab 所提供的更强大的机器,并耗费许多个小时**。

幸运的是,**我们已经在这个场景中训练好了一个智能体,它就发布在 🤗 Hub 上!** 让我们下载这个模型,看看智能体的表现。

In [ ]:
# 从 Hub 下载智能体
!python -m sample_factory.huggingface.load_from_hub -r edbeeching/doom_deathmatch_bots_2222 -d ./train_dir

由于智能体会游玩很长时间,视频的生成可能需要 **10 分钟**。

In [ ]:

from sample_factory.enjoy import enjoy
register_vizdoom_components()
env = "doom_deathmatch_bots"
cfg = parse_vizdoom_cfg(argv=[f"--env={env}", "--num_workers=1", "--save_video", "--no_render", "--max_num_episodes=1", "--experiment=doom_deathmatch_bots_2222", "--train_dir=train_dir"], evaluation=True)
status = enjoy(cfg)
mp4 = open('/content/train_dir/doom_deathmatch_bots_2222/replay.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)


你**可以尝试使用上面的代码在这个环境中训练你的智能体**,但不能在 Colab 上进行。
**祝你好运 🤞**

如果你想要一个更简单的场景,**为什么不试试在另一个 ViZDoom 场景中训练呢,比如 `doom_deadly_corridor`(致命走廊)或 `doom_defend_the_center`(坚守中心)。**



---


最后一个单元到此结束。但我们还没有结束!🤗 接下来的**附加(bonus)部分包含了深度强化学习领域一些最有趣、最进阶、最前沿的工作**。

## 持续学习,保持出色 🤗